In [1]:
import cobra
import sys
sys.path.append('../')
from modelfunctions import *
import os

# TODO remove and clean up
from memote.support.consistency import check_stoichiometric_consistency
import memote.support.consistency_helpers as con_helpers
from models_emil import * 

wd = os.path.abspath(os.getcwd()).removesuffix('Code/modelUpdates')
figdir = wd + 'Figures/'
models_dir = f'{wd}Data/pcm/'

In [2]:
def met_overview(pcm, rid):
    return pd.DataFrame({m.id:{'Formula': m.formula, 'factor': f, 'Charge': m.charge, 'Name': m.name, 'nRxns': len(m.reactions)} for m,f in get_rid(pcm, rid).metabolites.items()}).T.sort_values('factor')

def check_num_imbalanced(pcm, balance_with_protons = False, return_imbalanced = False):
    c = 0
    #slime_reactions = []
    imbalanced = []
    for r in pcm.reactions:
        # skip import export exchane biomass balance
        if any(r.id.startswith(prefix) for prefix in ['Im_', 'Ex_', 'Exch_', 'DM_', 'Sk_']) or \
                r.id in ['PigmentPool', 'BiomassRxn', 'precursorPool', 'ProteinPool']:
            continue

        # These do not really count since they only supply the biomass reaction
        if r.id.startswith('SLIMEr'):
            #slime_reactions.append(r)
            continue

        try:
            imba = r.check_mass_balance()
        except ValueError as e:
            c += 1
            if return_imbalanced:
                imbalanced.append(r)
            continue

        # 'X' refers to photons, and they cannot be balanced
        # ignore charge imbalances for now
        if not set(imba.keys()).difference(['X']):
            continue

        if imba and not (balance_with_protons and set(imba.keys()) == {'H', 'charge'} and imba['H'] == imba['charge']):
            if return_imbalanced:
                imbalanced.append(r)
            c += 1
    if return_imbalanced:
        return c, imbalanced
    return c

In [138]:
check_num_imbalanced(pcm, return_imbalanced=True)

(1, [<Reaction R06961 at 0x7610094c59a0>])

In [3]:
pcm1 = cobra.io.read_sbml_model(models_dir + 'pcm.v1.xml')

Set parameter Username
Set parameter LicenseID to value 2852561
Academic license - for non-commercial use only - expires 2027-08-10


In [123]:
pcm = cobra.io.read_sbml_model(models_dir + 'pcm.v2.xml')

In [ ]:
# TODO
# Notes:
# deal with dead-end metabolites
# unblock all reactions
# check stoichiometric consistency

In [124]:
deadM = {m for m in pcm.metabolites if len(m.reactions) == 1}
rs = {r for m in deadM for r in m.reactions}

In [125]:
len(rs)

161

In [126]:
print_rxns(sorted(rs, key=lambda r:r.id))

R00187
H2O, water + P1,P3-Bis(5'-adenosyl) triphosphate --> Adenosine 5-monophosphate + Adenosine 5-diphosphate + 2.0 H+, proton
C00001[h] + C06197[h] --> AMP[h] + C00008[h] + 2.0 C00080[h]

R00191
H2O, water + 3',5'-Cyclic AMP --> Adenosine 5-monophosphate + 2.0 H+, proton
C00001[h] + C00575[h] --> AMP[h] + 2.0 C00080[h]

R00251
2.0 H2O, water + Adenosine 5-triphosphate + 5-Oxoproline --> Adenosine 5-diphosphate + Orthophosphate + Glutamate + 2.0 H+, proton
2.0 C00001[h] + C00002[h] + C01879[h] --> C00008[h] + C00009[h] + C00025[h] + 2.0 C00080[h]

R00264
H2O, water + Nicotinamide adenine dinucleotide phosphate + 2,5-Dioxopentanoate --> Nicotinamide adenine dinucleotide phosphate - reduced + alpha-Ketoglutarate, 2-Oxoglutarate + 3.0 H+, proton
C00001[h] + C00006[h] + C00433[h] --> C00005[h] + C00026[h] + 3.0 C00080[h]

R00293
UDP-Glucose + 2.0 H+, proton --> H2O, water + UDP-4-dehydro-6-deoxy-D-glucose
C00029[h] + 2.0 C00080[h] --> C00001[h] + C04089[h]

R00336
H2O, water + Guanosine 

In [144]:
met_overview(pcm, 'R09366')

,Formula,factor,Charge,Name,nRxns
C00001[h],H2O,-1.0,0,"H2O, water",395
C05689[h],C4H9NO2Se,-1.0,0,Se-Methyl-L-selenocysteine,1
NH4[h],H4N,1.0,1,Ammonia,52
C00022[h],C3H3O3,1.0,-1,Pyruvate,54
C05703[h],CH4Se,1.0,0,Methaneselenol,2


In [133]:
print_rxns_mid(pcm, 'C05703[h]', add_func = lambda r: print('\t'.join(str(len(m.reactions)) for m in r.metabolites)))

R09366: C00001[h] + C05689[h] --> C00022[h] + C05703[h] + NH4[h]
H2O, water + Se-Methyl-L-selenocysteine --> Pyruvate + Methaneselenol + Ammonia
395	1	52	54	2

R09372: 2.0 C00005[h] + 2.0 C00080[h] + C18902[h] <=> 2.0 C00001[h] + 2.0 C00006[h] + C05703[h]
2.0 Nicotinamide adenine dinucleotide phosphate - reduced + 2.0 H+, proton + Methylselenic acid <=> 2.0 H2O, water + 2.0 Nicotinamide adenine dinucleotide phosphate + Methaneselenol
110	730	1	395	110	2



In [ ]:
get_mid(pcm, 'C05689[h]')

Metabolite identifier,C05689[h]
Name,Se-Methyl-L-selenocysteine
Memory address,0x761009668b90
Formula,C4H9NO2Se
Compartment,h
In 1 reaction(s),R09366


In [ ]:
# TODO This is only relevant for hyperaccumulators:
# R09366 might be legit, if Methaneselenol C05703 is transformed to dimethyldiselenide, DMDSe
    # (e.g. https://doi.org/10.1039/b914255j)
# Then, need reaction to methylate SeCys (C05688) to MetSeCys C05689 (see R04931)
# Also the more usual route of DMSe is missing

In [ ]:
get_rid(pcm, 'R09366')

In [ ]:
# TODO check effect of Selenate import ub (none)
print(pcm.slim_optimize())
get_rid(pcm, 'Exch_SeO4').upper_bound = 100
print(pcm.slim_optimize())
get_rid(pcm, 'Exch_SeO4').upper_bound = 1000
print(pcm.slim_optimize())

0.2766098694401416
0.2766098694401416
0.2766098694401416


In [136]:
match_mname(pcm, 'selen')

C05684[h] Selenite
C05697[h] Selenate
C05697[c] Selenate
C05703[h] Methaneselenol
C05688[h] L-Selenocysteine
C01528[h] Hydrogen selenide
C05686[h] Adenylyl selenate
C05698[h] Selenohomocysteine
C18902[h] Methylselenic acid
C05698[c] Selenohomocysteine
C05699[h] L-Selenocystathionine
C05689[h] Se-Methyl-L-selenocysteine
C05696[h] 3'-Phosphoadenylyl selenate


[<Metabolite C05684[h] at 0x76100b969550>,
 <Metabolite C05697[h] at 0x7610096559d0>,
 <Metabolite C05697[c] at 0x7610096ac5f0>,
 <Metabolite C05703[h] at 0x761009668cb0>,
 <Metabolite C05688[h] at 0x76100b7341d0>,
 <Metabolite C01528[h] at 0x76100b888650>,
 <Metabolite C05686[h] at 0x761009655760>,
 <Metabolite C05698[h] at 0x761009655a30>,
 <Metabolite C18902[h] at 0x761009668f80>,
 <Metabolite C05698[c] at 0x761009679a90>,
 <Metabolite C05699[h] at 0x761009655b80>,
 <Metabolite C05689[h] at 0x761009668b90>,
 <Metabolite C05696[h] at 0x76100ac7b290>]

In [121]:
match_mname(pcm, 'selen')

C05684[h] Selenite
C05697[h] Selenate
C05703[h] Methaneselenol
C05688[h] L-Selenocysteine
C01528[h] Hydrogen selenide
C05686[h] Adenylyl selenate
C05698[h] Selenohomocysteine
C05335[h] L-Selenomethionine
C18902[h] Methylselenic acid
C05699[h] L-Selenocystathionine
C05689[h] Se-Methyl-L-selenocysteine
C05696[h] 3'-Phosphoadenylyl selenate


[<Metabolite C05684[h] at 0x76101062e780>,
 <Metabolite C05697[h] at 0x76101027d070>,
 <Metabolite C05703[h] at 0x76101064a210>,
 <Metabolite C05688[h] at 0x76101062e840>,
 <Metabolite C01528[h] at 0x761011e90860>,
 <Metabolite C05686[h] at 0x76101027cd40>,
 <Metabolite C05698[h] at 0x76101027d0d0>,
 <Metabolite C05335[h] at 0x761010284620>,
 <Metabolite C18902[h] at 0x761010284b30>,
 <Metabolite C05699[h] at 0x76101027d010>,
 <Metabolite C05689[h] at 0x761010284710>,
 <Metabolite C05696[h] at 0x76101027cef0>]

In [112]:
get_rid(pcm, 'ATPSL_h')

Reaction identifier,ATPSL_h
Name,ATP sulfurylase
Memory address,0x76100b705c10
Stoichiometry,"C00002[h] + C00080[h] + SO4[h] --> C00013[h] + C00224[h] Adenosine 5-triphosphate + H+, proton + Sulfate --> Diphosphate, Pyrophosphate + Adenosine 5-phosphosulfate"
GPR,At1g19920 or At3g22890 or At4g14680 or At5g43780
Lower bound,0.0
Upper bound,1000.0


In [62]:
rs_fully = []
for r in rs:
    if len([m for m in r.metabolites if len(m.reactions) == 1]) > 1:
        rs_fully.append(r)
len(rs_fully)

3

In [808]:
rs_fully

[<Reaction R03210 at 0x72206fd685f0>,
 <Reaction R04882 at 0x72207013a930>,
 <Reaction R03182 at 0x72206fcd49b0>,
 <Reaction R10783 at 0x72206ffd0c80>,
 <Reaction R03161 at 0x72206fc951c0>,
 <Reaction R04294 at 0x72206ff29490>,
 <Reaction R12172 at 0x72206feeb770>]

In [ ]:
with pcm:
    for r in pcm.reactions:
        if r.id.startswith('Im_'):
            pass#r.knock_out()
        elif r.id.startswith('Exch_'):
            pass#r.knock_out()
    res = check_production(pcm, 'H-Cys[h]')#, add_import=['C00073[h]'])
res